# KFUPM — Replacement Source: Pure Research Outputs (OAI-PMH)

**Saudi Technology Research Data Hub · KFUPM source**

This notebook extracts, cleans, and validates the **King Fahd University of Petroleum and Minerals (KFUPM)** records for the project.

> ⚠️ **This is a new dataset. It is different from the KFUPM data we used first.**

## Why we replaced the original KFUPM data

The first KFUPM source was **KFUPM ePrints**. It had two problems:

| Problem | Detail |
|---|---|
| **DOI problem** | The ePrints data had only **48 thesis records**. They were theses, not published research papers, so they did not have a reliable DOI that points to the research itself. For our project the DOI is a key field (quality checks, `has_doi`, matching with Crossref/OpenAlex), so these records were weak. |
| **Data no longer available** | The original ePrints data was removed, so we could not rebuild it. |

## The new source

We use **KFUPM Pure**, the university's official research portal (built on Elsevier Pure).

- Every record is a **published research output** (journal article, conference paper, review, chapter).
- Most records have a **real DOI stored inside Pure itself**. We do not need to look it up in an external API.
- The data is collected with **OAI-PMH**, the harvesting interface Pure provides for this purpose.
  - We do **not** scrape HTML pages: the portal pages block automated requests (HTTP 403) and reserve text/data-mining rights.
  - The OAI endpoint is public and needs **no API key**. (The Pure REST API at `/ws/api` needs authentication, so we do not use it.)

| Item | Old source | New source |
|---|---|---|
| Name | KFUPM ePrints | KFUPM Pure Research Outputs |
| Record type | Theses | Published research outputs |
| Records | 48 | **415** (after filtering) |
| DOI | Not reliable | Stored in Pure — **100% of final records** |
| Access | JSON file | OAI-PMH (XML / MODS), no authentication |
| Scope | Computer Engineering | Computer Engineering, 2023–2026 |

## Pipeline in this notebook

```
KFUPM Pure OAI-PMH endpoint
      ↓   1. Test the endpoint (Identify / formats / sets)
      ↓   2. Harvest publications for 2023–2026 (raw XML, unchanged)
      ↓   3. Inspect the raw data (department names, identifier types)
      ↓   4. Keep Department of Computer Engineering only
      ↓   5. Map MODS fields → 13-column common schema
      ↓   6. Filters: year range, DOI required, research genres only
      ↓   7. Profile the result
      ↓   8. Validate against schema rules
      ↓
data/interim/KFUPM_cleaned.csv
```

## 0. Setup

Imports, paths, and one helper for OAI requests.

The project root is detected automatically, so the notebook works whether it runs from the repository root or from the `notebooks/` folder.

In [1]:
import re
import json
import time
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

# Project root: the folder that contains data/ (works from repo root or notebooks/)
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw" / "kfupm_pure"
INTERIM_DIR = ROOT / "data" / "interim"
RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://pure.kfupm.edu.sa/ws/oai"
NS = {"oai": "http://www.openarchives.org/OAI/2.0/"}   # OAI-PMH namespace
M = "{http://www.loc.gov/mods/v3}"                    # MODS namespace

YEARS = range(2023, 2027)

print("Project root:", ROOT)
print("Raw folder:  ", RAW_DIR)

Project root: c:\Users\nawaf\saudi-tech-research
Raw folder:   c:\Users\nawaf\saudi-tech-research\data\raw\kfupm_pure


In [2]:
def oai(**params):
    """Send one OAI-PMH request and return the parsed XML root."""
    r = requests.get(BASE, params=params, timeout=60)
    r.raise_for_status()
    return ET.fromstring(r.content)

## 1. Test the OAI-PMH endpoint

Before writing the full extraction, we check three things:

1. **`Identify`**: the endpoint is alive and follows OAI-PMH 2.0.
2. **`ListMetadataFormats`**: which XML formats are offered.
3. **`ListSets`**: how records are grouped, so we can select only what we need.

In [3]:
root = oai(verb="Identify")
ident = root.find("oai:Identify", NS)
for tag in ["repositoryName", "baseURL", "protocolVersion",
            "earliestDatestamp", "deletedRecord", "granularity"]:
    print(f"{tag:18}: {ident.findtext(f'oai:{tag}', namespaces=NS)}")

repositoryName    : Pure OAI Repository
baseURL           : https://pure.kfupm.edu.sa/ws/oai
protocolVersion   : 2.0
earliestDatestamp : 2022-09-28T08:45:22Z
deletedRecord     : no
granularity       : YYYY-MM-DDThh:mm:ssZ


In [4]:
root = oai(verb="ListMetadataFormats")
print("Available metadata formats:")
for f in root.findall(".//oai:metadataFormat", NS):
    print(" -", f.findtext("oai:metadataPrefix", namespaces=NS))

Available metadata formats:
 - mods
 - mods_swepub
 - xmetadiss
 - oai_dc
 - nl_didl
 - qdc
 - uketd_dc


**Result:** the formats include `oai_dc`, `mods`, `qdc`, and others.

We use **`mods`** (MODS 3.8) because it is richer than `oai_dc`. It stores the DOI, the journal, the publication type, and the organisational units in separate, typed fields.

In [5]:
all_sets, params = [], {"verb": "ListSets"}
while True:
    root = oai(**params)
    for s in root.findall(".//oai:set", NS):
        all_sets.append((s.findtext("oai:setSpec", namespaces=NS),
                         s.findtext("oai:setName", namespaces=NS)))
    tok = root.find(".//oai:resumptionToken", NS)
    if tok is None or not (tok.text or "").strip():
        break
    params = {"verb": "ListSets", "resumptionToken": tok.text.strip()}
    time.sleep(1)

print("Total sets:", len(all_sets))
print("\nNon-year sets:")
for spec, name in all_sets:
    if ":year" not in spec:
        print(f" - {spec:30} | {name}")

print("\nSets mentioning 'comput':",
      [s for s in all_sets if "comput" in (s[1] or "").lower()])

Total sets: 199

Non-year sets:
 - persons:all                    | PURE Persons
 - publications:all               | PURE publications
 - publications:withFiles         | PURE publications (with associated files)
 - openaire                       | OpenAIRE
 - studenttheses:all              | PURE theses
 - studenttheses:withFiles        | PURE theses (with associated files)
 - datasets:all                   | PURE DataSets

Sets mentioning 'comput': []


**Result:**

- There are **year sets**: `publications:year2023` … `publications:year2026`. We can harvest only the years we need.
- There are **no department sets**. So we harvest all publications for 2023–2026 and filter on the department **inside each record** (step 4).

In [6]:
print("Records per year set:")
for y in YEARS:
    root = oai(verb="ListIdentifiers", metadataPrefix="mods",
               set=f"publications:year{y}")
    tok = root.find(".//oai:resumptionToken", NS)
    n = tok.get("completeListSize") if tok is not None \
        else len(root.findall(".//oai:header", NS))
    print(f"  {y}: {n}")
    time.sleep(1)

Records per year set:
  2023: 3433
  2024: 4015
  2025: 4881
  2026: 3927


**Result:** about **16,256 records** for the whole university (2023: 3,433 · 2024: 4,015 · 2025: 4,881 · 2026: 3,927). This is a reasonable size to harvest.

## 2. Harvest the raw data (Extract)

We download every page of `ListRecords` for each year set and save it **exactly as the server returned it** in `data/raw/kfupm_pure/`. This follows the project rule: *raw source files must remain unchanged*.

How the harvest works:

- **Paging:** OAI-PMH returns about 100 records per page. Each page gives a `resumptionToken` for the next page.
- **Retries:** the server sometimes returns **HTTP 502 (Bad Gateway)** during long harvests. Each request is retried up to 6 times with a longer wait each time.
- **Checkpoint:** after each page, the next token is saved. If the harvest stops, re-running the cell continues from the same page.
- **Skip finished years:** a `yearYYYY.done` file marks a completed year, so re-running the notebook does not download it again.
- **Polite pacing:** 1.5 seconds between requests.

> The full harvest takes about **10–12 minutes**.
>
> If a harvest is resumed much later, the server may reply `badResumptionToken` because tokens expire. In that case, delete that year's `page` files and `checkpoint` file, then run the cell again.

In [7]:
def get_with_retry(params, retries=6):
    for i in range(retries):
        try:
            r = requests.get(BASE, params=params, timeout=120)
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            wait = 10 * (i + 1)
            print(f"  retry {i + 1}/{retries} in {wait}s:", str(e)[:80])
            time.sleep(wait)
    raise RuntimeError("Request failed after retries")


for year in YEARS:
    done = RAW_DIR / f"year{year}.done"
    ckpt = RAW_DIR / f"year{year}.checkpoint.json"
    if done.exists():
        print(year, "already harvested -> skipping")
        continue

    if ckpt.exists():                                   # resume
        state = json.loads(ckpt.read_text())
        page, params = state["page"], state["params"]
        print(year, "resuming from page", page)
    else:                                               # fresh start
        page = 0
        params = {"verb": "ListRecords", "metadataPrefix": "mods",
                  "set": f"publications:year{year}"}

    while True:
        content = get_with_retry(params)
        (RAW_DIR / f"year{year}_page{page:04d}.xml").write_bytes(content)  # raw, unchanged
        root = ET.fromstring(content)
        tok = root.find(".//oai:resumptionToken", NS)
        if tok is None or not (tok.text or "").strip():
            break
        page += 1
        params = {"verb": "ListRecords", "resumptionToken": tok.text.strip()}
        ckpt.write_text(json.dumps({"page": page, "params": params}))
        if page % 10 == 0:
            print(f"  {year}: page {page}, cursor {tok.get('cursor')}/{tok.get('completeListSize')}")
        time.sleep(1.5)

    total = sum(len(ET.parse(f).getroot().findall(".//oai:record", NS))
                for f in RAW_DIR.glob(f"year{year}_page*.xml"))
    done.write_text(str(total))
    ckpt.unlink(missing_ok=True)
    print(year, "->", total, "records in", page + 1, "pages")

  2023: page 10, cursor 900/3433
  2023: page 20, cursor 1900/3433
  2023: page 30, cursor 2900/3433
2023 -> 3433 records in 35 pages
  2024: page 10, cursor 900/4015
  2024: page 20, cursor 1900/4015
  2024: page 30, cursor 2900/4015
  2024: page 40, cursor 3900/4015
2024 -> 4015 records in 41 pages
  2025: page 10, cursor 900/4881
  2025: page 20, cursor 1900/4881
  2025: page 30, cursor 2900/4881
  2025: page 40, cursor 3900/4881
2025 -> 4881 records in 49 pages
  2026: page 10, cursor 900/3927
  2026: page 20, cursor 1900/3927
  2026: page 30, cursor 2900/3927
2026 -> 3927 records in 40 pages


In [8]:
# Check the harvest against the expected set sizes
summary = {y: int((RAW_DIR / f"year{y}.done").read_text()) for y in YEARS}
print("Harvested records per year:", summary)
print("Total:", sum(summary.values()))
print("Raw XML files:", len(list(RAW_DIR.glob("*.xml"))))

Harvested records per year: {2023: 3433, 2024: 4015, 2025: 4881, 2026: 3927}
Total: 16256
Raw XML files: 165


**Result:** 2023 → 3,433 · 2024 → 4,015 · 2025 → 4,881 · 2026 → 3,927, in 165 XML pages. The totals match the set sizes from step 1, so nothing was lost. The 502 errors were recovered by the retry logic.

## 3. Inspect the raw data (Profile)

Before mapping any fields, we look at the raw MODS to answer:

1. **How is the department stored?** As an organisational unit (`name type="corporate"`), or only as free-text author `affiliation`?
2. **Which identifier types exist?** Is `doi` present, and how often?
3. **How are people and organisations separated?** (the `name` types and roles)

This step only reads the saved files. It does not contact the server.

In [9]:
corp, affil, id_types, roles = Counter(), Counter(), Counter(), Counter()

for f in sorted(RAW_DIR.glob("*.xml")):
    root = ET.parse(f).getroot()
    for m in root.iter(f"{M}mods"):
        for n in m.iter(f"{M}name"):
            role = " / ".join(t.text or "" for t in n.iter(f"{M}roleTerm"))
            roles[(n.get("type"), role)] += 1
            if n.get("type") == "corporate":
                for p in n.findall(f"{M}namePart"):
                    corp[(p.text or "").strip()] += 1
            for a in n.findall(f"{M}affiliation"):
                affil[(a.text or "").strip()] += 1
        for i in m.findall(f"{M}identifier"):
            id_types[i.get("type")] += 1

print("Name types / roles:")
for k, v in roles.most_common(8):
    print(f"  {v:6}  {k}")

print("\nIdentifier types:")
for k, v in id_types.most_common():
    print(f"  {v:6}  {k}")

print("\nCorporate (organisational unit) names containing 'comput':")
for k, v in corp.most_common():
    if "comput" in k.lower():
        print(f"  {v:6}  {k}")

print("\nFree-text affiliations containing 'comput' (top 10):")
for k, v in [x for x in affil.most_common() if "comput" in x[0].lower()][:10]:
    print(f"  {v:6}  {k}")

Name types / roles:
   42251  ('personal', 'author')
   15868  ('corporate', 'pbl / /dk/atira/pure/organisation/organisationtypes/organisation/center')
   13392  ('corporate', 'pbl / /dk/atira/pure/organisation/organisationtypes/organisation/department')
    8214  ('personal', 'author / correspondingAuthor')
    1708  ('personal', 'editor')
    1105  ('personal', 'inventor')
     924  ('corporate', 'pbl / /dk/atira/pure/organisation/organisationtypes/organisation/university / 60009506 / 122667920 / 60091167 / 128145790 / 60212277 / 126521517 / 126798389 / 126751402 / 126644413 / 127146754 / 128043203 / 127148489 / 127022346 / 127404837 / 126530248 / 126499902 / 127402666 / 127875210 / 127933110 / 126521505')
     283  ('corporate', 'pbl / /dk/atira/pure/organisation/organisationtypes/organisation/college')

Identifier types:
   16256  pure/id
   16256  pure/uuid
   16256  uri
   15245  scopus
   14891  doi
    2565  isbn
     967  patent_number
     939  local
     633  pmid

Corporate

**Findings:**

- **Department:** Pure stores KFUPM departments as organisational units (`name type="corporate"`). The exact name is **`Department of Computer Engineering`** (449 records).
  - We filter on this field only.
  - We do **not** use the free-text `affiliation` field. It is typed by authors and includes departments from **other universities** (for example, "Department of Electrical and Computer Engineering, Northwestern University").
- **DOI:** `doi` exists in **14,891 of 16,256** records (about 92%), so DOIs are stored directly in Pure.
- **Other identifiers:** every record has `pure/uuid` (a stable ID) and `uri` (the portal link).
- **Separate department:** *Department of Information and Computer Science* is a different department at KFUPM. It is **out of scope** for this source.

## 4–5. Filter the department and map to the common schema (Clean & Standardize)

Each MODS record is converted to the project's **13-column common schema**:

| Schema column | Taken from (MODS) | Rule |
|---|---|---|
| `research_id` | `identifier[@type='pure/uuid']` | Prefixed with `KFUPM_`; stable and unique |
| `university` | constant | King Fahd University of Petroleum and Minerals |
| `title` | `titleInfo` (no type) → `title` (+ `subTitle`) | Whitespace cleaned |
| `authors` | `name[@type='personal']` → given + family | Joined with `; ` |
| `publication_year` | `originInfo/dateIssued` | First 4 digits |
| `publication_date` | `originInfo/dateIssued` | **Only** if it is a full `YYYY-MM-DD`; otherwise empty (no guessing) |
| `abstract` | `abstract` | Whitespace cleaned |
| `research_field` | `subject/topic` | Joined with `; ` when available |
| `tech_category` | — | Empty; assigned later in the project |
| `journal` | `relatedItem[@type='host']/titleInfo/title` | Journal or conference name |
| `doi` | `identifier[@type='doi']` | Normalized to bare `10.xxxx/...` form, lowercase |
| `url` | `identifier[@type='uri']` | Portal page, `https://pure.kfupm.edu.sa/en/publications/<uuid>` |
| `source` | constant | `KFUPM Pure (OAI-PMH)` |

Two helper columns, `_genre` and `_date_raw`, are kept **only for checking**. They are removed before saving.

In [10]:
DEPT = "Department of Computer Engineering"
UNIVERSITY = "King Fahd University of Petroleum and Minerals"
SOURCE = "KFUPM Pure (OAI-PMH)"
DOI_RE = re.compile(r"10\.\d{4,9}/\S+", re.I)


def txt(el):
    """Text of an element with whitespace collapsed (None if missing)."""
    if el is None:
        return None
    value = re.sub(r"\s+", " ", "".join(el.itertext())).strip()
    return value or None


def parse_record(mods):
    # --- department filter (organisational unit, exact match)
    org_units = [txt(p) for n in mods.findall(f"{M}name[@type='corporate']")
                 for p in n.findall(f"{M}namePart")]
    if DEPT not in org_units:
        return None

    ids = {}
    for i in mods.findall(f"{M}identifier"):
        ids.setdefault(i.get("type"), txt(i))

    # --- title (main titleInfo has no type attribute)
    ti = next((t for t in mods.findall(f"{M}titleInfo") if t.get("type") is None), None)
    title = txt(ti.find(f"{M}title")) if ti is not None else None
    subtitle = txt(ti.find(f"{M}subTitle")) if ti is not None else None
    if title and subtitle:
        title = f"{title}: {subtitle}"

    # --- authors
    authors = []
    for n in mods.findall(f"{M}name[@type='personal']"):
        given = " ".join(filter(None, (txt(p) for p in n.findall(f"{M}namePart[@type='given']"))))
        family = " ".join(filter(None, (txt(p) for p in n.findall(f"{M}namePart[@type='family']"))))
        full = f"{given} {family}".strip() or txt(n.find(f"{M}namePart"))
        if full:
            authors.append(full)

    # --- dates: never invent missing month/day
    date_raw = txt(mods.find(f".//{M}originInfo/{M}dateIssued"))
    year = int(date_raw[:4]) if date_raw and date_raw[:4].isdigit() else None
    full_date = date_raw if date_raw and re.fullmatch(r"\d{4}-\d{2}-\d{2}", date_raw) else None

    # --- DOI: keep only the bare DOI, lowercase
    doi = None
    if ids.get("doi"):
        match = DOI_RE.search(ids["doi"])
        doi = match.group(0).rstrip(".").lower() if match else None

    # --- URL: portal link, fallback to location/url
    url = ids.get("uri")
    if not (url and url.startswith(("http://", "https://"))):
        url = txt(mods.find(f".//{M}location/{M}url"))

    topics = [txt(t) for t in mods.findall(f".//{M}subject/{M}topic")]

    return {
        "research_id": f"KFUPM_{ids.get('pure/uuid')}",
        "university": UNIVERSITY,
        "title": title,
        "authors": "; ".join(authors) or None,
        "publication_year": year,
        "publication_date": full_date,
        "abstract": txt(mods.find(f"{M}abstract")),
        "research_field": "; ".join(t for t in topics if t) or None,
        "tech_category": None,
        "journal": txt(mods.find(f"{M}relatedItem[@type='host']/{M}titleInfo/{M}title")),
        "doi": doi,
        "url": url,
        "source": SOURCE,
        "_genre": txt(mods.find(f"{M}genre")),   # helper, removed before saving
        "_date_raw": date_raw,                   # helper, removed before saving
    }

In [11]:
rows = []
for f in sorted(RAW_DIR.glob("*.xml")):
    for mods in ET.parse(f).getroot().iter(f"{M}mods"):
        record = parse_record(mods)
        if record:
            rows.append(record)

df = pd.DataFrame(rows)
print("Computer Engineering records:", len(df))
print("Duplicate research_id:", df["research_id"].duplicated().sum())
df = df.drop_duplicates("research_id")

Computer Engineering records: 449
Duplicate research_id: 0


**Result:** **449** Computer Engineering records, with no duplicate IDs.

## 6. Apply the source filters

| Filter | Reason | Effect |
|---|---|---|
| `publication_year` between 2023 and 2026 | Project year range for this source | 449 → 449 |
| `doi` is not null | **The main reason for this new source.** Every KFUPM record should point to a real research DOI. | 449 → 423 (26 removed) |
| Remove `Editorial` and `Comment/debate` | Not research outputs | 423 → **415** (8 removed) |

> **Note:** requiring a DOI is a decision **for the KFUPM source only**. In the common schema, `doi` stays **nullable** because other sources may have valid records without a DOI.

In [12]:
EXCLUDED_GENRES = ["Editorial", "Comment/debate"]

n0 = len(df)
df = df[df["publication_year"].between(2023, 2026)]
n1 = len(df)
df = df[df["doi"].notna()]
n2 = len(df)
print("Genres before genre filter:")
print(df["_genre"].value_counts().to_string(), "\n")
df = df[~df["_genre"].isin(EXCLUDED_GENRES)]
n3 = len(df)

print(f"Start:                    {n0}")
print(f"After year filter:        {n1}  (-{n0 - n1})")
print(f"After DOI filter:         {n2}  (-{n1 - n2})")
print(f"After genre filter:       {n3}  (-{n2 - n3})")
print("Duplicate DOI:", df["doi"].duplicated().sum())

Genres before genre filter:
_genre
Article                    294
Conference contribution     85
Review article              16
Conference article          15
Editorial                    7
Chapter                      4
Comment/debate               1
Paper                        1 

Start:                    449
After year filter:        449  (-0)
After DOI filter:         423  (-26)
After genre filter:       415  (-8)
Duplicate DOI: 0


## 7. Profile the cleaned data

In [13]:
print("Records per year:")
print(df["publication_year"].value_counts().sort_index().to_string())

print("\nGenres:")
print(df["_genre"].value_counts().to_string())

print("\nDate formats in source (9 = any digit):")
print(df["_date_raw"].str.replace(r"\d", "9", regex=True).value_counts().to_string())

print("\nMissing values (%):")
print((df.isna().mean() * 100).round(1).to_string())

Records per year:
publication_year
2023     77
2024     79
2025    149
2026    110

Genres:
_genre
Article                    294
Conference contribution     85
Review article              16
Conference article          15
Chapter                      4
Paper                        1

Date formats in source (9 = any digit):
_date_raw
9999          204
9999-99       130
9999-99-99     81

Missing values (%):
research_id           0.0
university            0.0
title                 0.0
authors               0.0
publication_year      0.0
publication_date     80.5
abstract              0.5
research_field       66.3
tech_category       100.0
journal               0.0
doi                   0.0
url                   0.0
source                0.0
_genre                0.0
_date_raw             0.0


In [14]:
df[["research_id", "title", "authors", "publication_year", "journal", "doi", "url"]].head()

,research_id,title,authors,publication_year,journal,doi,url
0,KFUPM_c954706c-188f-4ae4-91b6-54cddf544465,Heat transmission in Darcy-Forchheimer flow of...,Ambreen A. Khan; Alina Arshad; R. Ellahi; Sadi...,2023,International Journal of Numerical Methods for...,10.1108/hff-03-2022-0194,https://pure.kfupm.edu.sa/en/publications/c954...
1,KFUPM_3d94024a-b56a-471c-b3fa-cc06a68ed1d2,Are mega-events super spreaders of infectious ...,Tamal Chowdhury; Hemal Chowdhury; Elza Bontemp...,2023,Environmental Science and Pollution Research,10.1007/s11356-022-22660-2,https://pure.kfupm.edu.sa/en/publications/3d94...
2,KFUPM_718632be-16ea-4617-ac51-03e44082747d,Genetic algorithm optimization to model busine...,Khaled A. Al Utaibi; Robia Arif; Sadiq M. Sait...,2023,International Journal of Management Science an...,10.1080/17509653.2022.2076169,https://pure.kfupm.edu.sa/en/publications/7186...
3,KFUPM_a6621fce-0a4b-489a-a886-02a1441c86ad,Natural convection nanofluid flow with heat tr...,Rahmat Ellahi; Ahmed Zeeshan; Aamir Waheed; Na...,2023,Mathematical Methods in the Applied Sciences,10.1002/mma.7281,https://pure.kfupm.edu.sa/en/publications/a662...
4,KFUPM_fa1bbebf-be43-4d6a-8f75-2c9caa682d7b,Lightweight Two-Factor-Based User Authenticati...,Alawi A. Al-saggaf; Tarek Sheltami; Hoda Alkhz...,2023,Arabian Journal for Science and Engineering,10.1007/s13369-022-07235-0,https://pure.kfupm.edu.sa/en/publications/fa1b...


**Findings:**

- **Per year:** 2023 → 79 · 2024 → 80 · 2025 → 153 · 2026 → 111 before the genre filter (423 records); the output above shows the final counts.
- **Genres:** mostly `Article` and `Conference contribution`, plus reviews, conference articles, and chapters.
- **`publication_date` is about 80% empty.** This is expected and correct. The source gives only the year (204 records) or year-month (130 records) for most items. Only 81 records have a full date. Following the project rule, **we do not invent missing months or days**.
- **Required fields** (`research_id`, `university`, `title`, `publication_year`, `url`, `source`) are **0% missing**. `doi` is also 0% missing after the filter.
- **`research_field`** is about 66% empty and **`tech_category`** is 100% empty. Both are optional, and `tech_category` is assigned later.
- **Some DOIs contain "2022"** (e.g. `10.1108/hff-03-2022-0194`). This is normal: the paper was accepted or published online in 2022 and assigned to a 2023 journal issue.
- **Cross-disciplinary papers:** some titles are outside computing (e.g. fluid dynamics, public health). They are joint papers that include a Computer Engineering faculty member, so they are valid for this department. They need care when assigning `tech_category`.

## 8. Validate against the common schema and save

The checks below follow the project's schema rules. If any check fails, the cell stops with an error, and nothing is saved.

In [15]:
SCHEMA_COLUMNS = [
    "research_id", "university", "title", "authors", "publication_year",
    "publication_date", "abstract", "research_field", "tech_category",
    "journal", "doi", "url", "source",
]
REQUIRED = ["research_id", "university", "title", "publication_year", "url", "source"]

final = df.drop(columns=["_genre", "_date_raw"])[SCHEMA_COLUMNS].copy()
final["publication_year"] = final["publication_year"].astype("Int64")

assert list(final.columns) == SCHEMA_COLUMNS, "Columns do not match the schema"
assert final["research_id"].is_unique, "Duplicate research_id"
assert final[REQUIRED].notna().all().all(), "Missing value in a required field"
assert (final["title"].str.strip() != "").all(), "Empty title"
assert final["publication_year"].between(2023, 2026).all(), "Year out of range"
assert final["url"].str.match(r"^https?://").all(), "Invalid URL"
assert final["doi"].notna().all(), "Missing DOI (required for KFUPM)"
assert final["doi"].str.match(r"^10\.\d{4,9}/\S+$").all(), "Invalid DOI format"
assert pd.to_datetime(final["publication_date"].dropna(),
                      format="%Y-%m-%d", errors="coerce").notna().all(), "Invalid publication_date"
assert (final["university"] == UNIVERSITY).all(), "Non-standard university name"

print("All validation checks passed ✅")
print("Shape:", final.shape)

All validation checks passed ✅
Shape: (415, 13)


In [16]:
out_path = INTERIM_DIR / "KFUPM_cleaned.csv"
final.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved:", out_path)

Saved: c:\Users\nawaf\saudi-tech-research\data\interim\KFUPM_cleaned.csv


## Summary

| Stage | Records |
|---|---|
| Harvested (all KFUPM publications, 2023–2026) | 16,256 |
| Department of Computer Engineering | 449 |
| With DOI | 423 |
| Research genres only (final) | **415** |

**Outputs**

- Raw data: `data/raw/kfupm_pure/year{YYYY}_page{NNNN}.xml` (unchanged OAI-PMH responses)
- Clean data: `data/interim/KFUPM_cleaned.csv` (13-column common schema)

**Compared with the old KFUPM source:** we moved from **48 thesis records** with unreliable DOIs to **415 published research outputs**, and **every record has a real DOI** taken from Pure.

**Documentation to update**

- README "Data Sources" table: *KFUPM Pure Research Outputs · Institutional Research Repository · XML (OAI-PMH / MODS) · None · Computer Engineering research records for 2023–2026*
- Requirements workbook: new URL (`https://pure.kfupm.edu.sa/ws/oai`), format, size, and known issues (occasional HTTP 502 during harvesting, about 80% of records without a full publication date).